# 01 — Coding Assistant Architecture

This notebook explores how coding assistants work — from user request to tool execution to final response.

## Learning Objectives
1. Understand the tool use loop that powers all coding assistants
2. Build a simple tool executor in Python
3. Simulate a complete tool use conversation with Claude

## Setup

In [ ]:
import os
import json
from pathlib import Path
from dotenv import load_dotenv

load_dotenv()

# Verify API key is available
api_key = os.getenv("ANTHROPIC_API_KEY")
assert api_key, "Set ANTHROPIC_API_KEY in your .env file"
print("API key loaded successfully.")

## The Architecture

```
User Request
     ↓
Language Model (Claude)
     ↓
Tool Call Decision (read_file, write_file, run_command)
     ↓
Tool Executor (application layer)
     ↓
Result fed back to Claude
     ↓
Final Response to User
```

LLMs only produce text. Tools bridge the gap between text and action.

## Building a Tool Executor

Let's build a minimal tool executor that handles `read_file`, `write_file`, and `list_dir`.

In [ ]:
import subprocess
from typing import Any


def execute_tool(tool_name: str, tool_input: dict[str, Any]) -> str:
    """Execute a tool call and return the result as text."""
    match tool_name:
        case "read_file":
            path = Path(tool_input["file_path"])
            if not path.exists():
                return f"Error: File not found: {path}"
            return path.read_text()

        case "write_file":
            path = Path(tool_input["file_path"])
            path.parent.mkdir(parents=True, exist_ok=True)
            path.write_text(tool_input["content"])
            return f"File written: {path}"

        case "list_dir":
            path = Path(tool_input.get("path", "."))
            if not path.is_dir():
                return f"Error: Not a directory: {path}"
            entries = sorted(p.name for p in path.iterdir())
            return "\n".join(entries)

        case "run_command":
            result = subprocess.run(
                tool_input["command"],
                shell=True,
                capture_output=True,
                text=True,
                timeout=30,
            )
            output = result.stdout
            if result.returncode != 0:
                output += f"\nSTDERR: {result.stderr}"
            return output

        case _:
            return f"Unknown tool: {tool_name}"


print("Tool executor defined.")

## Testing the Tool Executor

In [ ]:
# Test read_file
result = execute_tool("read_file", {"file_path": "../requirements.txt"})
print("=== read_file ===")
print(result)

# Test list_dir
result = execute_tool("list_dir", {"path": ".."})
print("\n=== list_dir ===")
print(result)

# Test run_command
result = execute_tool("run_command", {"command": "echo 'Hello from tool executor'"})
print("\n=== run_command ===")
print(result)

## Defining Tools for Claude

Claude uses the Anthropic API's tool definitions to understand what tools are available.

In [ ]:
TOOLS = [
    {
        "name": "read_file",
        "description": "Read the contents of a file from disk.",
        "input_schema": {
            "type": "object",
            "properties": {
                "file_path": {
                    "type": "string",
                    "description": "The path to the file to read.",
                }
            },
            "required": ["file_path"],
        },
    },
    {
        "name": "list_dir",
        "description": "List the contents of a directory.",
        "input_schema": {
            "type": "object",
            "properties": {
                "path": {
                    "type": "string",
                    "description": "The directory path to list. Defaults to current directory.",
                }
            },
            "required": [],
        },
    },
    {
        "name": "run_command",
        "description": "Execute a shell command and return its output.",
        "input_schema": {
            "type": "object",
            "properties": {
                "command": {
                    "type": "string",
                    "description": "The shell command to execute.",
                }
            },
            "required": ["command"],
        },
    },
]

print(f"Defined {len(TOOLS)} tools: {[t['name'] for t in TOOLS]}")

## Complete Tool Use Loop

Now let's wire it all together: send a request to Claude with tools, handle tool calls, and get the final answer.

In [ ]:
import anthropic

client = anthropic.Anthropic()


def coding_assistant(user_request: str, max_iterations: int = 10) -> str:
    """Run a complete tool use loop with Claude."""
    messages = [{"role": "user", "content": user_request}]

    for i in range(max_iterations):
        response = client.messages.create(
            model="claude-sonnet-4-20250514",
            max_tokens=4096,
            system="You are a coding assistant. Use the available tools to help the user.",
            tools=TOOLS,
            messages=messages,
        )

        print(f"\n--- Iteration {i + 1} (stop_reason: {response.stop_reason}) ---")

        # If Claude is done, extract the text
        if response.stop_reason == "end_turn":
            text_blocks = [b.text for b in response.content if b.type == "text"]
            return "\n".join(text_blocks)

        # Process tool calls
        if response.stop_reason == "tool_use":
            messages.append({"role": "assistant", "content": response.content})

            tool_results = []
            for block in response.content:
                if block.type == "tool_use":
                    print(f"  Tool: {block.name}({json.dumps(block.input)})")
                    result = execute_tool(block.name, block.input)
                    print(f"  Result: {result[:200]}..." if len(result) > 200 else f"  Result: {result}")
                    tool_results.append({
                        "type": "tool_result",
                        "tool_use_id": block.id,
                        "content": result,
                    })

            messages.append({"role": "user", "content": tool_results})

    return "Max iterations reached."


print("Coding assistant function defined.")

In [ ]:
# Run the coding assistant
result = coding_assistant("What files are in the parent directory? Read the requirements.txt file.")
print("\n=== FINAL RESPONSE ===")
print(result)

## Exercise

1. Add a `write_file` tool to the TOOLS list and update the coding assistant to handle it
2. Ask the assistant to create a new file based on the contents of an existing one
3. Add error handling for tool execution failures